## G-eval

In [1]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams, LLMTestCase

criteria = """Coherence (1-5) - the collective quality of all sentences. We align this dimension with
the DUC quality question of structure and coherence whereby the summary should be
well-structured and well-organized. The summary should not just be a heap of related information, but should build from sentence to sentence to a coherent body of information about a topic."""

coherence_metric = GEval(
    name="Coherence",
    criteria=criteria,
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT, LLMTestCaseParams.EXPECTED_OUTPUT],
)

# Now define your test case, actual_output is your LLM output
test_case = LLMTestCase(input="Hey how's the weather like today?", actual_output="It's alright!")

# Use G-Eval metric
coherence_metric.measure(test_case)
print(coherence_metric.score, coherence_metric.reason)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [2]:
import numpy as np 
import json
import os
import shutil
import subprocess

from sacrebleu.metrics import BLEU
from rouge import Rouge

from evaluate import load

from sentence_transformers import CrossEncoder

from transformers import AutoModelForCausalLM, AutoTokenizer

bleu_scorer = BLEU()
rouge_scorer = Rouge()
bertscore = load("bertscore")

ModuleNotFoundError: No module named 'sacrebleu'

In [6]:
def compute_rouge(hypothesis, reference):
    """
    Returns the prediction and the reference for a single dialogue, returns their ROUGE score.
    """
    score = rouge_scorer.get_scores(
          hyps=hypothesis,
          refs=reference,
      )
    return score[0]["rouge-l"]["f"]

def compute_bleu(hypothesis, reference):
    """
    Returns the prediction and the reference for a single dialogue, returns their BLEU score.
    """
    score = bleu_scorer.sentence_score(
        hypothesis=hypothesis,
        references=[reference],
      )
    return score.score/100 # sacreBLEU gives the score in percent

def compute_bertscore(predictions, references):
    """
    Receives two lists of strings with equal length. Returns their pairwise Bertscores (precision, recall, F1 score).
    """
    results = bertscore.compute(predictions=predictions, references=references, lang="en")
    return results

In [3]:
_qwen3_model = None
_qwen3_tokenizer = None

_qwen2_model = None
_qwen2_tokenizer = None


def get_qwen3_model():
    global _qwen3_model, _qwen3_tokenizer
    if _qwen3_model is None or _qwen3_tokenizer is None:
        model_name = "Qwen/Qwen3-1.7B"
        _qwen3_tokenizer = AutoTokenizer.from_pretrained(model_name)
        _qwen3_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
    return _qwen3_model, _qwen3_tokenizer


def get_qwen2_model():
    global _qwen2_model, _qwen2_tokenizer
    if _qwen2_model is None or _qwen2_tokenizer is None:
        model_name = "Qwen/Qwen3-1.7B"
        _qwen2_tokenizer = AutoTokenizer.from_pretrained(model_name)
        _qwen2_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype="auto",
            device_map="auto"
        )
    return _qwen2_model, _qwen2_tokenizer

In [4]:
def query_qwen_as_a_judge(messages, model_name):

    system_prompt = {"role":"system", "content": "Give a score from 0.0 to 1.0 based on how coherent is the prediction based on the ground truth, where 0.0 is not coherent at all and 1.0 is very coherent. Output just the score nothing else."}
    messages_judge = [system_prompt] + messages
    if model_name == "Qwen3-1.7B":
        model, tokenizer = get_qwen3_model()

        text = tokenizer.apply_chat_template(
        messages_judge,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False
        )
        
    elif model_name == "Qwen2.5-3B":
        model, tokenizer = get_qwen2_model()

        text = tokenizer.apply_chat_template(
        messages_judge,
        tokenize=False,
        add_generation_prompt=True
        )
        
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=2000
    )
    output_ids = generated_ids[0][model_inputs.input_ids.shape[1]:]
    output = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    return output

In [5]:

model_name = "Qwen3-1.7B"

llm_as_a_judge_output_qwen = []
for prediction, reference in zip(all_predictions, all_references):
    input_message = [{"role":"user", "content": f"Prediction: {prediction} \n Ground truth: {reference} \n Score: "}]
    result = query_qwen_as_a_judge(input_message, "Qwen3-1.7B")
    llm_as_a_judge_output_qwen.append(result)

NameError: name 'all_predictions' is not defined